In [ ]:
from buganize import Buganize

# Comments and Updates

This notebook demonstrates fetching comments and field change history for issues.

## Get Comments

Fetch comments for an issue in chronological order.

In [ ]:
async with Buganize() as client:
    result = await client.comments(486077869)

print(f"{len(result.comments)} comments (total: {result.total_count})")
for comment in result.comments:
    author = comment.author or "unknown"
    date = comment.timestamp.strftime("%Y-%m-%d %H:%M") if comment.timestamp else "?"
    print(f"#{comment.comment_number} by {author} ({date})")
    print(f"  {comment.body[:120]}")
    print()

## Reading Individual Comments

In [ ]:
for comment in result.comments[:5]:  # first 5
    author = comment.author or "unknown"
    date = comment.timestamp.strftime("%Y-%m-%d %H:%M") if comment.timestamp else "?"
    print(f"#{comment.comment_number} by {author} ({date})")
    print(f"  {comment.body[:120]}")
    print()

## Sort Order and Pagination

Comments support `"ASC"` (oldest first, default) and `"DESC"` (newest first). Use `page_size` and `page_token` to paginate.

In [ ]:
# Newest comments first
async with Buganize() as client:
    result = await client.comments(486077869, sort_order="DESC", page_size=3)

print(f"Page 1: {len(result.comments)} comments, has_more={result.has_more}")
for comment in result.comments:
    print(f"  #{comment.comment_number} — {comment.body[:80]}")

# Fetch next page
if result.has_more:
    page2 = await client.comments(
        486077869,
        sort_order="DESC",
        page_size=3,
        page_token=result.next_page_token,
    )
    print(f"\nPage 2: {len(page2.comments)} comments, has_more={page2.has_more}")
    for comment in page2.comments:
        print(f"  #{comment.comment_number} — {comment.body[:80]}")

## Full Updates (Comments + Field Changes)

The `issue_updates()` method returns both comments and field changes (status, priority, etc.).

In [ ]:
async with Buganize() as client:
    result = await client.issue_updates(486077869)

print(f"Total updates: {result.total_count}")
print(f"Updates fetched: {len(result.updates)}")
print(f"Comments: {len(result.comments)}")

In [ ]:
for update in result.updates[:10]:  # first 10 (newest first)
    author = update.author or "unknown"
    date = update.timestamp.strftime("%Y-%m-%d %H:%M") if update.timestamp else "?"
    parts = []
    if update.field_changes:
        fields = ", ".join(fc.field for fc in update.field_changes)
        parts.append(f"changed: {fields}")
    if update.comment:
        parts.append(f"comment #{update.comment.comment_number}")
    print(f"{author} ({date}) — {'; '.join(parts)}")